# Chapter 0: Anatomy & The Mathematics of Splitting

> **Why this chapter?** XGBoost, LightGBM, and Random Forest are all built on one primitive: the **Decision Tree node split**. If you don't deeply understand how a single split is chosen mathematically, the advanced models will always feel like a black box. This chapter is the foundation of everything that follows.

---

## 1. What is a Decision Tree?

### **The Core Idea**
A Decision Tree is a **supervised learning algorithm** that works like a flowchart of yes/no questions. Think of the game **"Guess Who?"** — you ask "Does your person wear glasses?" and eliminate half the candidates. Then "Is their hair dark?" and eliminate half again. With enough well-chosen questions, you can identify anyone.

A Decision Tree does *exactly* this with data:
- **"Is Income > $50k?"** → Yes → go left. No → go right.
- **"Is Age > 35?"** → (asked only to those who went left) → Yes/No → split again.
- Eventually you reach a final answer: **Predict YES** or **Predict NO**.

### **Two Types of Output (Critical Distinction)**
| Type | Target Variable | Leaf Predicts | Example |
| :--- | :--- | :--- | :--- |
| **Classification Tree** | Categorical (classes) | **Majority class** in that leaf | Spam / Not Spam |
| **Regression Tree** | Continuous (numbers) | **Mean value** of all samples in that leaf | House Price |

This distinction matters throughout the whole course — the *splitting math* differs between the two.

### **Tree Anatomy**
```
                   [Root Node]
                  Income > 50k?
                 /              \
           YES /                \ NO
              /                  \
    [Decision Node]           [Leaf Node]
     Age > 35?                Predict: REJECT
      /     \
    YES      NO
     /        \
[Leaf]       [Leaf]
APPROVE      REJECT
```
- **Root Node:** Contains the entire training set. The very first split is made here.
- **Decision Node (Internal Node):** Contains a subset of the data. Applies a threshold to one feature and sends data left or right.
- **Leaf Node (Terminal Node):** Contains a final group. No more questions. Outputs a prediction.

### **How Splits Look Geometrically**
Imagine plotting your data on 2D axes (Feature 1 = Income, Feature 2 = Age). Each split draws a **straight line parallel to one axis**:
- `Income > 50k` draws a **vertical line** at $x = 50,000$.
- `Age > 35` draws a **horizontal line** at $y = 35$.

Decision Trees can **only** draw **axis-aligned** (horizontal or vertical) lines. They cannot naturally draw diagonal boundaries like $Income = Age \times 1000$. This is a fundamental structural limitation we will revisit in Section 8.

---

## 2. How Does the Tree Choose a Split? (The Greedy Search)

### **The Problem**
At any given node, you have $F$ features and each feature has many possible threshold values. The tree must choose one (feature, threshold) pair. You cannot try all combinations globally — it would be computationally intractable. So the tree uses a **greedy algorithm**.

### **What "Greedy" Actually Means**
At each node, the tree asks: *"What single split, right now, reduces impurity the most?"* It picks that split. It does **not** look two or three levels ahead to see if a slightly worse split now could unlock a much better split later. It only optimizes the current node.

> **Consequence of Greedy:** Imagine Feature A gives IG = 0.4 and Feature B gives IG = 0.35. The tree picks A. But if Feature B were chosen first, the next level would have an IG of 0.9, resulting in a much better tree overall. The greedy tree misses this — it is **not globally optimal**. This is why we need ensembles (Random Forest, GBM) to compensate.

### **The Actual Search Algorithm (The Nested Loop)**
Here is what actually happens inside a Decision Tree at every single node:

```
best_gain = 0
best_feature = None
best_threshold = None

FOR each feature f in [Feature_1, Feature_2, ..., Feature_F]:
    Sort all unique values of f → [v1, v2, v3, ..., vN]
    
    FOR each consecutive midpoint t = (vi + vi+1) / 2 as a candidate threshold:
        Split data: LEFT = samples where f <= t
                    RIGHT = samples where f > t
        
        Compute: gain = impurity(parent) - weighted_impurity(LEFT, RIGHT)
        
        IF gain > best_gain:
            best_gain = gain
            best_feature = f
            best_threshold = t

Split the node on (best_feature, best_threshold)
```

### **Why does it try midpoints?**
If you have ages `[20, 25, 30, 35]`, the candidate thresholds are `[22.5, 27.5, 32.5]`. Using the midpoint between adjacent sorted values guarantees we cover every possible binary split of the data.

### **Computational Complexity**
At each node: $O(F \cdot N \log N)$ — because for each of the $F$ features, we sort $N$ samples ($O(N \log N)$) and scan through $N$ thresholds. Across all nodes in a tree of depth $d$: $O(F \cdot N \log N \cdot 2^d)$. This is why deep trees on large datasets are expensive.

---

## 3. Impurity Metric A: Entropy & Information Gain

To evaluate how "good" a split is, we need a mathematical definition of "purity". **Entropy** from Claude Shannon's Information Theory is one such definition.

### **A. Conceptual Definition**
Entropy measures **uncertainty** — how hard it is to predict the class of a randomly picked sample from that node.
- **A bucket of 100 Red balls:** Entropy = 0. No uncertainty. You know the next ball is Red.
- **A bucket of 50 Red / 50 Blue:** Entropy = Maximum. You have zero idea what's next.
- **A bucket of 80 Red / 20 Blue:** Entropy is *between* 0 and max. You're fairly confident it's Red, but not certain.

### **B. The Formula**
For a node $S$ containing samples from $C$ classes:
$$H(S) = -\sum_{i=1}^{C} p_i \cdot \log_2(p_i)$$

**Unpacking each component:**
- $p_i$ — The **proportion** of class $i$ in the node. If 6 out of 10 samples are "Yes", then $p_{Yes} = 0.6$.
- $\log_2$ — We use **base 2** because entropy is measured in **bits** (binary information units). Base-2 log of a probability tells you how many binary (yes/no) questions you need to identify that outcome.
- **Negative sign** — Since $0 \le p_i \le 1$, we have $\log_2(p_i) \le 0$ always. The negative sign flips it to a positive value. Note: by convention, $0 \cdot \log_2(0) = 0$ (pure node contributes 0 to entropy).

### **C. Behavior of the Function**
| Situation | Entropy | Why |
| :--- | :---: | :--- |
| 100% one class ($p = 1.0, 0.0$) | **0** | $-1 \cdot \log_2(1) = 0$. No uncertainty. |
| 50/50 two classes ($p = 0.5, 0.5$) | **1.0** | $-(0.5 \cdot (-1) + 0.5 \cdot (-1)) = 1.0$. Maximum for binary. |
| 25% each of 4 classes | **2.0** | $-(4 \times 0.25 \cdot (-2)) = 2.0$. Max for 4-class. |

**General Rule:** For $k$ equally distributed classes, Maximum Entropy $= \log_2(k)$.
- Binary ($k=2$): Max = $\log_2(2) = 1.0$
- 4-class ($k=4$): Max = $\log_2(4) = 2.0$  
- 8-class ($k=8$): Max = $\log_2(8) = 3.0$

This means raw entropy values **cannot be compared across datasets with different numbers of classes**!

### **D. Intuition: Entropy as a "Questions Game"**
Entropy tells you: *"On average, how many yes/no questions do I need to identify the class of a random sample from this node?"*
- **Pure Node (All Red):** 0 questions. You already know it's Red. Entropy = 0.
- **50/50 Red/Blue:** 1 question ("Is it Red?"). Entropy = 1.
- **25% each of Red/Blue/Green/Yellow:** Need 2 questions to narrow it down. Entropy = 2.

### **E. Information Gain: The Split Scoring Formula**
Entropy alone doesn't tell us which split to choose. **Information Gain (IG)** measures *how much entropy is reduced* by making a specific split.

$$IG(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} \cdot H(S_v)$$

**Translating each part:**
- $H(S)$ — Entropy of the **parent** node (before the split).
- $\frac{|S_v|}{|S|}$ — The **weight** of each child. A larger child has more influence. If the left child gets 60% of the samples and the right gets 40%, the left's entropy counts for 60%.
- $H(S_v)$ — Entropy of each **child** node (after the split).
- The subtraction: $IG = H_{before} - H_{after}$. How much disorder did we *remove*?

**Algorithm Decision:** Compute IG for every (feature, threshold) pair. Pick the one with the **highest IG**.

---

## 4. Impurity Metric B: Gini Impurity

### **A. Conceptual Definition (The "Mis-labeling" Game)**
Gini Impurity asks: *"If I randomly pick a sample from this node and randomly assign it a label according to the distribution of labels in the node, what is the probability I mislabel it?"*

- **Pure node (all Red):** I pick a sample, it's Red. I assign Red (since distribution is 100% Red). Probability of error = 0. Gini = 0.
- **50/50 node:** I pick a sample (50% Red). I assign a random label (50% Red). Probability of mislabeling is high. Gini = 0.5.

### **B. Mathematical Derivation (From First Principles)**
Let's prove the formula step by step.

Say we have two classes A (with probability $p_A$) and B (with probability $p_B = 1 - p_A$).

**Probability of a Match (correct labeling):**
- Pick a sample, it's class A (prob = $p_A$). Assign label A (prob = $p_A$). Both match: $p_A \times p_A = p_A^2$
- Pick a sample, it's class B (prob = $p_B$). Assign label B (prob = $p_B$). Both match: $p_B \times p_B = p_B^2$
- Total probability of a match: $p_A^2 + p_B^2 = \sum p_i^2$

**Probability of a Mismatch (this is Gini):**
$$Gini = 1 - P(Match) = 1 - \sum_{i=1}^{C} p_i^2$$

This is not just a formula handed to you — it *falls out directly* from the definition of mislabeling probability. This is the derivation an interviewer might ask for.

### **C. Gini Range and the Multi-class Trap**

A very common interview trick question: *"Is the maximum Gini always 0.5?"* — **No.**

The max Gini depends on the number of classes:

| Classes ($k$) | Most Impure State | Max Gini |
| :---: | :--- | :---: |
| 2 | 50% each | $1 - (0.5^2 + 0.5^2) = 0.5$ |
| 4 | 25% each | $1 - (4 \times 0.25^2) = 1 - 0.25 = 0.75$ |
| 10 | 10% each | $1 - (10 \times 0.1^2) = 1 - 0.1 = 0.9$ |

**General Rule:** Maximum Gini for $k$ classes $= 1 - \frac{1}{k}$

As $k$ increases, maximum impurity approaches 1 (but never reaches it exactly).

### **D. Gini Gain (The Split Scoring Formula)**
Just like IG for Entropy, we compute **Gini Gain** to evaluate a split:
$$GiniGain(S, A) = Gini(S) - \sum_{v} \frac{|S_v|}{|S|} \cdot Gini(S_v)$$

The formula structure is **identical** to Information Gain — only the inner impurity function changes.

---

## ✍️ 5. Side-by-Side Worked Example: Gini vs. Entropy (Same Dataset)

Let's compute BOTH metrics on the exact same split and confirm they arrive at the same conclusion. This is the most important exercise in this chapter.

**Dataset:** 10 students, Target: Passed Exam? (6 Yes, 4 No)
**Feature being tested:** `Studied > 5 hours?`
- **Left Child (Studied > 5 hrs):** 5 Yes, 1 No — Total = 6
- **Right Child (Studied ≤ 5 hrs):** 1 Yes, 3 No — Total = 4

---

### **PART A: Gini Impurity Calculation**

**Step 1 — Parent Gini:**
$$Gini_{Parent} = 1 - \left(\left(\frac{6}{10}\right)^2 + \left(\frac{4}{10}\right)^2\right) = 1 - (0.36 + 0.16) = 1 - 0.52 = \mathbf{0.48}$$

**Step 2 — Left Child Gini** (5Y, 1N out of 6):
$$Gini_{Left} = 1 - \left(\left(\frac{5}{6}\right)^2 + \left(\frac{1}{6}\right)^2\right) = 1 - \left(\frac{25}{36} + \frac{1}{36}\right) = 1 - \frac{26}{36} \approx \mathbf{0.278}$$

**Step 3 — Right Child Gini** (1Y, 3N out of 4):
$$Gini_{Right} = 1 - \left(\left(\frac{1}{4}\right)^2 + \left(\frac{3}{4}\right)^2\right) = 1 - (0.0625 + 0.5625) = 1 - 0.625 = \mathbf{0.375}$$

**Step 4 — Weighted Gini of Children:**
$$Weighted\_Gini = \frac{6}{10} \times 0.278 + \frac{4}{10} \times 0.375 = 0.1668 + 0.150 = \mathbf{0.3168}$$

**Step 5 — Gini Gain:**
$$GiniGain = 0.48 - 0.3168 = \mathbf{0.1632}$$

---

### **PART B: Entropy & Information Gain Calculation (Same Split)**

**Step 1 — Parent Entropy:**
$$H_{Parent} = -\left(\frac{6}{10} \log_2 \frac{6}{10} + \frac{4}{10} \log_2 \frac{4}{10}\right)$$
$$= -(0.6 \times (-0.737) + 0.4 \times (-1.322))$$
$$= -(-0.4422 - 0.5288) = \mathbf{0.971 \text{ bits}}$$

**Step 2 — Left Child Entropy** (5Y, 1N out of 6):
$$H_{Left} = -\left(\frac{5}{6} \log_2 \frac{5}{6} + \frac{1}{6} \log_2 \frac{1}{6}\right)$$
$$= -(0.833 \times (-0.263) + 0.167 \times (-2.585))$$
$$= -(-0.219 - 0.432) = \mathbf{0.650 \text{ bits}}$$

**Step 3 — Right Child Entropy** (1Y, 3N out of 4):
$$H_{Right} = -\left(\frac{1}{4} \log_2 \frac{1}{4} + \frac{3}{4} \log_2 \frac{3}{4}\right)$$
$$= -(0.25 \times (-2) + 0.75 \times (-0.415))$$
$$= -(-0.5 - 0.311) = \mathbf{0.811 \text{ bits}}$$

**Step 4 — Weighted Entropy of Children:**
$$Weighted\_Entropy = \frac{6}{10} \times 0.650 + \frac{4}{10} \times 0.811 = 0.390 + 0.324 = \mathbf{0.714 \text{ bits}}$$

**Step 5 — Information Gain:**
$$IG = H_{Parent} - Weighted\_Entropy = 0.971 - 0.714 = \mathbf{0.257 \text{ bits}}$$

---

### **Key Takeaway from This Exercise**
| Metric | Parent Impurity | Weighted Child Impurity | Gain | Winner? |
| :--- | :---: | :---: | :---: | :---: |
| Gini | 0.48 | 0.3168 | 0.1632 | ✅ Same split chosen |
| Entropy | 0.971 bits | 0.714 bits | 0.257 bits | ✅ Same split chosen |

Both metrics agree on which feature is worth splitting. The numbers look different, but the **relative ranking** of features is almost always the same. This is why we use the computationally cheaper Gini.

**What happens next?** The algorithm repeats this for ALL other features (Attendance, Previous Grades, etc.) and picks the one with the HIGHEST Gain. That feature becomes the split at this node. Then the process recurses on both child nodes.

---

## 6. Gini vs. Entropy: Full Comparison

| Feature | Entropy | Gini Impurity |
| :--- | :--- | :--- |
| **Formula** | $-\sum p_i \log_2 p_i$ | $1 - \sum p_i^2$ |
| **Range (Binary)** | $[0,\; 1.0]$ | $[0,\; 0.5]$ |
| **Range (k-class)** | $[0,\; \log_2 k]$ | $[0,\; 1 - 1/k]$ |
| **Computation** | Slow — requires $\log$ (expensive CPU op) | Fast — only squaring & arithmetic |
| **Algorithm** | ID3, C4.5 | CART (Scikit-Learn default) |
| **Multi-class Behavior** | Entropy grows with classes (no fixed ceiling) | Max is bounded; captures less nuance per class |
| **When Entropy wins** | Many classes, imbalanced data | Not applicable (Gini is simpler choice) |
| **Sensitivity** | More penalizes extreme probabilities (log curve) | More uniform penalty via squared differences |

### **The Scaling Argument (Why Industry Uses Gini)**
When you run a **Random Forest with 500 trees** on a dataset with 1 million rows and 100 features:
- Each tree has up to thousands of nodes.
- At each node, you test hundreds of thresholds per feature.
- Each test calls the impurity function.
- **Total calls: hundreds of millions.**

Each $\log_2$ call versus each $p^2$ call — that difference multiplied by hundreds of millions is real latency. Gini wins purely on engineering pragmatics.

---

## 7. The High-Cardinality Trap & The Gain Ratio Fix

### **Problem: Information Gain is Biased**
Information Gain has a fatal flaw: it naturally **favors features with many unique values** (high cardinality).

**Concrete Example:**
Dataset: 10 students. Target: Passed? Feature 1: `Studied>5hrs`. Feature 2: `Student_ID` (values 1–10).

If the tree splits on `Student_ID`:
- Node 1: Student ID=1 → 1 person → 100% pure. $H = 0$.
- Node 2: Student ID=2 → 1 person → 100% pure. $H = 0$.
- ... (10 nodes, all pure)
- Weighted entropy of children = $\sum \frac{1}{10} \times 0 = 0$
- $IG = H_{Parent} - 0 = H_{Parent}$ → **Maximum possible gain!**

The tree will **eagerly choose Student_ID** as the best split. But this is useless — it has only memorized individual IDs. It completely fails on unseen data.

### **The Root Cause**
Information Gain rewards any feature that creates pure buckets, without penalizing features that create *too many* or *too small* buckets. A feature with 1,000 unique values can create 1,000 pure buckets trivially.

### **Solution: Gain Ratio (C4.5 Algorithm)**
C4.5 introduced a **penalty term** called **Split Information** that measures how *fragmented* the split is.

#### **Split Information Formula:**
$$SplitInfo_A(S) = -\sum_{v \in Values(A)} \frac{|S_v|}{|S|} \cdot \log_2 \frac{|S_v|}{|S|}$$

⚠️ **Critical Distinction — Entropy vs. SplitInfo:**

Both formulas look identical: $-\sum p \log_2 p$. The difference is **what "p" represents**:

| Formula | What $p$ measures | What it captures |
| :--- | :--- | :--- |
| **Entropy $H(S)$** | Proportion of **class labels** (Yes/No) in the node | How mixed the *answers* are |
| **SplitInfo** | Proportion of **data rows** that go into each branch | How fragmented the *split* is |

**Intuition:**
- Entropy says: *"The 10-way ID split gave me pure kids! Score: Excellent."*
- SplitInfo says: *"But you created 10 tiny branches of 1 sample each. Score: Horrible (SplitInfo is very high)."*

#### **Gain Ratio Formula:**
$$GainRatio(S, A) = \frac{InformationGain(S, A)}{SplitInfo_A(S)}$$

Back to `Student_ID` example:
- $IG$ is huge (numerator large) ✅
- $SplitInfo$ is also huge (10 branches, each 1/10 the data: $-10 \times \frac{1}{10} \log_2 \frac{1}{10} = \log_2 10 \approx 3.32$)
- $GainRatio = \frac{Large}{Very\; Large} \approx$ small — **ID gets penalized!** ✅

For a 2-way split like `Studied>5hrs`:
- SplitInfo = $-(0.6 \log_2 0.6 + 0.4 \log_2 0.4) \approx 0.971$ — much smaller.
- GainRatio is higher for the valid feature. ✅

---

## 8. Critical Terminology (For Interview — Explained with Consequences)

### **Pure Node**
A node where $Gini = 0$ or $H = 0$. It contains only one class. No further splitting is needed.

### **Greedy Algorithm (and Why It Can Fail)**
The tree always picks the locally best split at each node. It never backtracks or looks ahead.

> **Example of Greedy Failure:** Suppose Feature A gives IG = 0.5, but using Feature A makes future splits worthless. Feature B gives IG = 0.3 now but enables a perfect split (IG = 0.9) at the next level. The greedy tree always picks A, missing the globally better path. This is why ensembles — Random Forest (averaging many greedy trees) and Gradient Boosting (correcting residuals of each greedy tree) — are needed.

### **Recursive Partitioning**
After a split, each child node becomes the new "parent" and the process repeats. The tree grows depth-first, left-first. Stopping criteria (min samples, max depth) terminate the recursion.

### **Axis-Aligned Splits and Their Limitation**
Every split checks **one feature** against **one threshold**: `if income > 50000`. On a 2D plot, this is a line parallel to one axis. The tree cannot natively create a diagonal boundary like `income + 2*age > 100`.

> **Consequence:** If the true decision boundary in your data is diagonal, the Decision Tree must approximate it with a "staircase" of many horizontal/vertical cuts. A very deep tree can approximate it, but at the cost of overfitting. This is why trees struggle with data that has strong linear dependencies between multiple features simultaneously.

### **Impurity vs. Purity (The vocabulary)**
- **High Impurity** = Mixed classes in a node = Bad (we want to reduce this).
- **High Purity** = One dominant class = Good (this is the goal).
- All splitting metrics (Gini, Entropy) measure **impurity**. We minimize impurity by choosing splits that maximize **gain** (reduction in impurity).

---

## 9. Chapter Summary & Interview Checklist

Before moving to Chapter 1, you should be able to confidently answer all of the following:

| # | Question | Depth Expected |
| :- | :--- | :--- |
| 1 | What is the difference between a Classification and Regression tree? | Leaf outputs: Majority class vs. Mean. |
| 2 | How exactly does the tree decide *which* feature and *which* threshold to split on? | The full nested greedy search loop. |
| 3 | What does "Greedy" mean in the context of Decision Trees, and what's its consequence? | Local optimal ≠ global optimal. Needs ensembles. |
| 4 | Write the Entropy formula and explain every symbol. | $H = -\sum p_i \log_2 p_i$, explain $p_i$, base 2, negative sign. |
| 5 | Write the Information Gain formula. | $IG = H(Parent) - \sum \frac{|S_v|}{|S|} H(S_v)$. |
| 6 | Write the Gini formula and derive it from probability. | Derive from $P(mismatch) = 1 - \sum p_i^2$. |
| 7 | What is the maximum Gini for a $k$-class problem? | $1 - 1/k$. |
| 8 | Why does Scikit-Learn use Gini by default? | Speed: $p^2$ vs. $\log p$, millions of calls in ensembles. |
| 9 | What is the high-cardinality problem with Information Gain? | ID-style features get maximum IG trivially. |
| 10 | What is Gain Ratio and how does it fix the problem? | $GainRatio = IG / SplitInfo$. SplitInfo penalizes fragmented splits. |
| 11 | What is the difference between Entropy and SplitInfo? | Same formula but $p$ represents class labels vs. branch sizes. |
| 12 | Why can't Decision Trees draw diagonal boundaries? | Each split checks one feature — produces axis-aligned cuts only. |

---

## 10. Implementation Playground

The cells below are structured to implement a **complete** toy decision tree splitting mechanism from scratch. Fill them in after studying this chapter. The goal is to verify your understanding by matching the manual calculations from Section 5.


In [ ]:
import numpy as np

# ─── PART 1: Core Impurity Functions ──────────────────────────────────────────

def calculate_gini(y):
    """
    Input:  y — array of class labels (e.g., [1,1,0,1,0])
    Output: float — Gini impurity of this node
    Expected: calculate_gini([1,1,1,1,1,0]) ≈ 0.278
    """
    # TODO: count each class, compute p_i, return 1 - sum(p_i^2)
    pass

def calculate_entropy(y):
    """
    Input:  y — array of class labels
    Output: float — Entropy of this node in bits
    Expected: calculate_entropy([1,1,1,1,1,0]) ≈ 0.650
    """
    # TODO: count each class, compute p_i, return -sum(p_i * log2(p_i))
    # Note: handle p_i = 0 case (0 * log(0) = 0 by convention)
    pass

In [ ]:
# ─── PART 2: Gain Functions ────────────────────────────────────────────────────

def information_gain(y_parent, y_left, y_right):
    """
    Compute IG = H(parent) - weighted_avg_H(children)
    Expected using worked example: IG ≈ 0.257 bits
    """
    # TODO
    pass

def gini_gain(y_parent, y_left, y_right):
    """
    Compute GiniGain = Gini(parent) - weighted_avg_Gini(children)
    Expected using worked example: GiniGain ≈ 0.1632
    """
    # TODO
    pass

# Verify against manual calculation from Section 5
y_parent = [1,1,1,1,1,1,0,0,0,0]  # 6 Yes, 4 No
y_left   = [1,1,1,1,1,0]           # 5 Yes, 1 No  (Studied > 5 hrs)
y_right  = [1,0,0,0]               # 1 Yes, 3 No  (Studied <= 5 hrs)

print(f"Information Gain  (expected ≈ 0.257): {information_gain(y_parent, y_left, y_right)}")
print(f"Gini Gain         (expected ≈ 0.163): {gini_gain(y_parent, y_left, y_right)}")

In [ ]:
# ─── PART 3: The Greedy Split Finder ──────────────────────────────────────────

def find_best_split(X, y, criterion='gini'):
    """
    Implements the nested loop greedy search described in Section 2.
    
    Input:
        X         — 2D array of features, shape (n_samples, n_features)
        y         — 1D array of labels, shape (n_samples,)
        criterion — 'gini' or 'entropy'
    
    Output:
        best_feature_idx  — index of the feature to split on
        best_threshold    — threshold value to split on
        best_gain         — the gain achieved by this split
    """
    # TODO:
    # 1. Init best_gain = 0, best_feature = None, best_threshold = None
    # 2. For each feature index f:
    #    a. Get sorted unique values of X[:, f]
    #    b. For each consecutive midpoint t:
    #       - Split: left = y[X[:, f] <= t], right = y[X[:, f] > t]
    #       - Compute gain using gini_gain or information_gain
    #       - Update best if gain is higher
    # 3. Return best_feature, best_threshold, best_gain
    pass

print("Find best split placeholder ready — implement the nested loop from Section 2!")